In [1]:
import pandas as pd
import numpy as np
from IPython.display import display
import sys, os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
%matplotlib inline
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from functions.transformers import *

In [2]:
#Crear copia de dataset crudo
df_train_raw = pd.read_csv('../data/raw/train.csv').drop('Id', axis=1)
df_train = df_train_raw.copy()
#df_train = df_train.drop('SalePrice', axis=1)  # labels deleted 
print(f'start dimensions --> {df_train_raw.shape}')
df_train_raw.head()

start dimensions --> (1460, 80)


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
#Analisis de datos faltantes
missing_data = df_train_raw.isnull().sum()
missing_data = pd.DataFrame(missing_data[missing_data > 0], columns=['missing_count'])
missing_data['missing_percentage (%)'] = np.round(missing_data['missing_count'] / df_train_raw.shape[0] * 100, 2)
missing_data = missing_data.sort_values(by='missing_count', ascending=False)
print('fields with missing data:')
missing_data

fields with missing data:


,missing_count,missing_percentage (%)
PoolQC,1453,99.52
MiscFeature,1406,96.30
Alley,1369,93.77
Fence,1179,80.75
MasVnrType,872,59.73
FireplaceQu,690,47.26
LotFrontage,259,17.74
GarageType,81,5.55
GarageYrBlt,81,5.55
GarageFinish,81,5.55


In [ ]:
#Manejo de datos faltantes
#Paso inicial eliminar las columnas que tienen mas del 85% de datos faltantes

#LotFrontage: La longitud lineal (en pies) de la calle que bordea la propiedad.
#Alley: El tipo de acceso al callejón.
#MasVnrType: El tipo de revestimiento de mampostería.
#FireplaceQu: La calidad de la chimenea.
#PoolQC: La calidad de la piscina.
#Fence: La calidad de la valla.
#MiscFeature: Una característica miscelánea no cubierta en otras categorías.

print(f'start dimentions {df_train.shape}')
initial_columns = df_train.columns
df_train = df_train.dropna(thresh=len(df_train) * 0.85, axis=1)
columns_deleted = [col for col in initial_columns if col not in df_train.columns]
print(f'columns deleted ({len(columns_deleted)}) --> {columns_deleted}')

In [ ]:
#Columnas con datos faltantes resultantes de la eliminacion anterior
#Definir en una funcion
def missing_data_report(df):
    missing_res_deleted = df.isnull().sum()
    missing_res_deleted = missing_res_deleted[missing_res_deleted > 0].sort_values(ascending=True)
    return missing_res_deleted
res = missing_data_report(df_train)
res

In [ ]:
#imputacion simple a: Electrical y MasVnrArea
#Electrical: El sistema eléctrico. -> Valores faltantes imputados con la moda
df_train['Electrical'] = df_train['Electrical'].fillna(df_train['Electrical'].mode()[0])
#MasVnrArea: Área de revestimiento de mampostería en pies cuadrados. -> Valores faltantes imputados con (0)
df_train['MasVnrArea'] = df_train['MasVnrArea'].fillna(0)

In [ ]:
#Se imputan bajo esta condicion a N/A -> no tiene sótano
#Se analiza graficamente todas las features que tienen como valores N/A -> no tiene sótano -> (TotalBsmtSF = 0 & BsmtUnfSF = 0)
cond_not_basement = (df_train['TotalBsmtSF'] == 0) & (df_train['BsmtUnfSF'] == 0)
#BsmtCond: Evalúa la condición general del sótano. -> (NA) no tiene sótano
#BsmtQual: Evalúa la altura del sótano. -> (NA) no tiene sótano
##BsmtFinType1: Evaluación del tipo de acabado del sótano. -> (NA) no tiene sótano
#BsmtExposure: Refleja la cantidad de exposición al sótano al aire exterior. -> (NA) no tiene sótano
#BsmtFinType2: Evaluación del tipo de acabado del sótano (si hay dos tipos). -> (NA) no tiene sótano
cols_not_basement = ['BsmtCond', 'BsmtQual', 'BsmtFinType1', 'BsmtExposure', 'BsmtFinType2']
df_train.loc[cond_not_basement, cols_not_basement] = df_train.loc[cond_not_basement, cols_not_basement].fillna('NA')
fig, ax = plt.subplots(ncols=2, nrows=3)
fig.set_size_inches(15,8)

axisOne = sns.violinplot(x='BsmtCond', y='TotalBsmtSF', data=df_train, cut=0, ax=ax[0][0])
axisOne.set_title('Distribución de TotalBsmtSF por BsmtCond')
axisOne.set_xlabel('BsmtCond')
axisOne.set_ylabel('TotalBsmtSF')
axisOne.grid(axis='y', linestyle='--');

axisTwo = sns.violinplot(x='BsmtQual', y='TotalBsmtSF', data=df_train, cut=0, ax=ax[0][1])
axisTwo.set_title('Distribución de TotalBsmtSF por BsmtQual')
axisTwo.set_xlabel('BsmtQual')
axisTwo.set_ylabel('TotalBsmtSF')
axisTwo.grid(axis='y', linestyle='--'); 

axisThree = sns.violinplot(x='BsmtFinType1', y='TotalBsmtSF', data=df_train, cut=0, ax=ax[1][0])
axisThree.set_title('Distribución de TotalBsmtSF por BsmtFinType1')
axisThree.set_xlabel('BsmtFinType1')
axisThree.set_ylabel('TotalBsmtSF')
axisThree.grid(axis='y', linestyle='--');

axisFour = sns.violinplot(x='BsmtExposure', y='TotalBsmtSF', data=df_train, cut=0, ax=ax[1][1])
axisFour.set_title('Distribución de TotalBsmtSF por BsmtExposure')
axisFour.set_xlabel('BsmtExposure')
axisFour.set_ylabel('TotalBsmtSF')
axisFour.grid(axis='y', linestyle='--');

axisFive = sns.violinplot(x='BsmtFinType2', y='TotalBsmtSF', data=df_train, cut=0, ax=ax[2][0])
axisFive.set_title('Distribución de TotalBsmtSF por BsmtFinType2')
axisFive.set_xlabel('BsmtFinType2')
axisFive.set_ylabel('TotalBsmtSF')
axisFive.grid(axis='y', linestyle='--');

fig.delaxes(ax[2][1])
plt.tight_layout()

In [ ]:
#Columnas con datos faltantes resultantes de la imputacion anterior

missing_data2 = df_train.isnull().sum().filter(like='Bsmt')
missing_data2 = pd.DataFrame(missing_data2[missing_data2 > 0], columns=['missing_count'])
display(missing_data2)

#BsmtExposure: Refleja la cantidad de exposición al sótano al aire exterior. (x)
#BsmtFinType2: Evaluación del tipo de acabado del sótano (si hay dos tipos). (x)
df_train['BsmtExposure'] = df_train['BsmtExposure'].fillna(df_train.groupby(['BsmtCond', 'BsmtFinType1', 'BsmtFinType2', 'BsmtQual', 'BsmtFullBath', 'BsmtHalfBath'])['BsmtExposure']
                                                    .transform(lambda x: x.mode()[0] if not x.mode().empty else 'No'))
df_train['BsmtFinType2'] = df_train['BsmtFinType2'].fillna(df_train.groupby(['BsmtCond', 'BsmtQual', 'BsmtFullBath', 'BsmtHalfBath', 'BsmtExposure'])['BsmtFinType2']
                                                    .transform(lambda x: x.mode()[0] if not x.mode().empty else 'Unf'))

In [ ]:
#Imputacion de datos faltantes en columnas relacionadas con el garage ['GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond'] -> ['GarageArea', 'GarageCars']
#GarageType ->  Ubicación del garaje. (NA) no tiene garage
#GarageYrBlt -> Año de construcción del garaje. (0) no tiene garage
#GarageFinish -> Acabado interior del garaje. (NA) no tiene garage
#GarageQual -> Calidad del garaje. (NA) no tiene garage
#GarageCond -> Condición del garaje. (NA) no tiene garage
list_categorical_not_garage = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']
list_numerical_not_garage = ['GarageYrBlt']
cond_not_garage = df_train['GarageArea'] == 0
df_train.loc[cond_not_garage, list_categorical_not_garage] = df_train.loc[cond_not_garage, list_categorical_not_garage].fillna('NA')
df_train.loc[cond_not_garage, list_numerical_not_garage] = df_train.loc[cond_not_garage, list_numerical_not_garage].fillna(0)

In [ ]:
fig, ax = plt.subplots(ncols=2, nrows=2)
fig.set_size_inches(15,8)
axisType = sns.violinplot(x='GarageType', y='GarageArea', data=df_train, cut=0, ax=ax[0][0])
axisType.set_title('Distribución del área del garaje por tipo de garaje')
axisType.set_xlabel('GarageType')
axisType.set_ylabel('GarageArea')
axisType.grid(axis='y', linestyle='--');
axisFinish = sns.violinplot(x='GarageFinish', y='GarageArea', data=df_train, cut=0, ax=ax[0][1])
axisFinish.set_title('Distribución del área del garaje por acabado del garaje')
axisFinish.set_xlabel('GarageFinish')
axisFinish.set_ylabel('GarageArea')
axisFinish.grid(axis='y', linestyle='--');
axisQual = sns.violinplot(x='GarageQual', y='GarageArea', data=df_train, cut=0, ax=ax[1][0])
axisQual.set_title('Distribución del área del garaje por calidad del garaje')
axisQual.set_xlabel('GarageQual')
axisQual.set_ylabel('GarageArea')
axisQual.grid(axis='y', linestyle='--');
axisCond = sns.violinplot(x='GarageCond', y='GarageArea', data=df_train, cut=0, ax=ax[1][1])
axisCond.set_title('Distribución del área del garaje por condición del garaje')
axisCond.set_xlabel('GarageCond')
axisCond.set_ylabel('GarageArea')
axisCond.grid(axis='y', linestyle='--');
plt.tight_layout()


In [ ]:
#Feature engineering
print(f'Numero de features --> {len(df_train.columns)}')
numeric_features = df_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df_train.select_dtypes(include=[object]).columns.tolist()
print(f'numericas {len(numeric_features)} --> {numeric_features}')
print(f'categoricas {len(categorical_features)} --> {categorical_features}')

In [ ]:
#MSZoning: Clasificación general de la zona de uso del suelo.
print('---'*20)
mszoning_cat_all = ['A', 'C', 'FV', 'I', 'RH', 'RL', 'RP', 'RM']
mszoning_cat_train = df_train['MSZoning'].unique().tolist()
missing_cats_mszoning = list(set(mszoning_cat_all) - set(mszoning_cat_train))
print(f'{"🟢" if len(missing_cats_mszoning) == 0 else "🔴"} missing categories {missing_cats_mszoning}')

mszzoning_new_cat = df_train[~df_train['MSZoning'].isin(mszoning_cat_all)]
mszzoning_new_cat = mszzoning_new_cat['MSZoning'].unique().tolist()
print(f'{"🟢" if len(mszzoning_new_cat) == 0 else "🔴"} new categories or diferent name -> {mszzoning_new_cat}')


#Considerar valores categoricos de MSZoning que no tienen ejemplos en el set de entrenamiento
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding

#1. Se reemplaza 'C (all)' por 'C'
df_train['MSZoning'] = df_train['MSZoning'].replace('C (all)', 'C')

mszoning_cat_after_engineering = df_train['MSZoning'].unique().tolist()
missing_cats_mszoning = list(set(mszoning_cat_all) - set(mszoning_cat_after_engineering))
print(f'{"🟢" if len(missing_cats_mszoning) == 0 else "🔴"} missing categories after engineering {missing_cats_mszoning}')

print('🟡 MSZoning: --> One-Hot-Encoding')
print('---'*20)

In [ ]:
#Street: Tipo de acceso a la calle.
print('---'*20)

street_cat_all = ['Grvl', 'Pave']
street_cat_train = df_train['Street'].unique().tolist()
missing_cats_street = list(set(street_cat_all) - set(street_cat_train))
print(f'{"🟢" if len(missing_cats_street) == 0 else "🔴"} missing categories {missing_cats_street}')

street_new_cat = df_train[~df_train['Street'].isin(street_cat_all)]
street_new_cat = street_new_cat['Street'].unique().tolist()
print(f'{"🟢" if len(street_new_cat) == 0 else "🔴"} new categories or diferent name -> {street_new_cat}')
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#@@ Podria ser ordinal encoding
print('🟡 Street: --> One-Hot-Encoding')
print('---'*20)

In [ ]:
#LotShape: Forma de la parcela.
print('---'*20)
lotshape_cat_all = ['Reg', 'IR1', 'IR2', 'IR3']
lotshape_cat_train = df_train['LotShape'].unique().tolist()
missing_cats_lotShape = list(set(lotshape_cat_all) - set(lotshape_cat_train))
print(f'{"🟢" if len(missing_cats_lotShape) == 0 else "🔴"} missing categories {missing_cats_lotShape}')

lotshape_new_cat = df_train[~df_train['LotShape'].isin(lotshape_cat_all)]
lotshape_new_cat = lotshape_new_cat['LotShape'].unique().tolist()
print(f'{"🟢" if len(lotshape_new_cat) == 0 else "🔴"} new categories or diferent name -> {lotshape_new_cat}')

#Se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias tienen un orden definido, se procede a hacer ordinal-encoding
#Preliminarmene se hara un mapeo estandar
print('🟡 LotShape: --> Ordinal Encoding')
mapping_lotshape = {'Reg': 0, 'IR1': 1, 'IR2': 2, 'IR3': 3}
display(mapping_lotshape)
print('---'*20)

In [ ]:
#LandContour: Plano de la parcela.
print('---'*20)
landcontour_cat_all = ['Lvl', 'Bnk', 'HLS', 'Low']
landcontour_cat_train = df_train['LandContour'].unique().tolist()
missing_cats_landcontour = list(set(landcontour_cat_all) - set(landcontour_cat_train))
print(f'{"🟢" if len(missing_cats_landcontour) == 0 else "🔴"} missing categories {missing_cats_landcontour}')

landcontour_new_cat = df_train[~df_train['LandContour'].isin(landcontour_cat_all)]
landcontour_new_cat = landcontour_new_cat['LandContour'].unique().tolist()
print(f'{"🟢" if len(landcontour_new_cat) == 0 else "🔴"} new categories or diferent name -> {landcontour_new_cat}')
#Se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias tienen un orden definido, se procede a hacer ordinal-encoding
print('🟡 LandContour: --> Ordinal Encoding')
mapping_landcontour = {'Lvl': 0, 'Bnk': 1, 'HLS': 2, 'Low': 3}
display(mapping_landcontour)
print('---'*20)

In [ ]:
#Utilities: Tipo de servicios públicos disponibles.
print('---'*20)
utilities_cat_all = ['AllPub', 'NoSewr', 'NoSeWa', 'ELO']
utilities_cat_train = df_train['Utilities'].unique().tolist()
missing_cats_utilities = list(set(utilities_cat_all) - set(utilities_cat_train))
print(f'{"🟢" if len(missing_cats_utilities) == 0 else "🔴"} missing categories {missing_cats_utilities}')

utilities_new_cat = df_train[~df_train['Utilities'].isin(utilities_cat_all)]
utilities_new_cat = utilities_new_cat['Utilities'].unique().tolist()
print(f'{"🟢" if len(utilities_new_cat) == 0 else "🔴"} new categories or diferent name -> {utilities_new_cat}')

#Se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias tienen un orden definido, se procede a hacer ordinal-encoding
#@@@: Puede ser probada con one-hot-encoding
print('🟡 Utilities: --> Ordinal Encoding')
mapping_utilities = {'AllPub': 3, 'NoSewr': 2, 'NoSeWa': 1, 'ELO': 0}
display(mapping_utilities)
print('---'*20)

In [ ]:
#LotConfig: Configuración de la parcela.
print('---'*20)
print('LotConfig: --> one hot encoding')
lot_config_cat_all = ['Inside', 'Corner', 'CulDSac', 'FR2', 'FR3']
lot_config_cat_train = df_train['LotConfig'].unique().tolist()
missing_cats_lotConfig = list(set(lot_config_cat_all) - set(lot_config_cat_train))
print(f'{"🟢" if len(missing_cats_lotConfig) == 0 else "🔴"} missing categories {missing_cats_lotConfig}')

lotconfig_new_cat = df_train[~df_train['LotConfig'].isin(lot_config_cat_all)]
lotconfig_new_cat = lotconfig_new_cat['LotConfig'].unique().tolist()
print(f'{"🟢" if len(lotconfig_new_cat) == 0 else "🔴"} new categories or diferent name -> {lotconfig_new_cat}')

#No tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#@@@: Puede ser probada con ordinal-encoding
print('🟡 LotConfig: --> One-Hot-Encoding')
print('---'*20)

In [ ]:
#LandSlope: Inclinación del terreno.
print('---'*20)
landslope_cat_all = ['Gtl', 'Mod', 'Sev']
landslope_cat_train = df_train['LandSlope'].unique().tolist()
missing_cats_landSlope = list(set(landslope_cat_all) - set(landslope_cat_train))
print(f'{"🟢" if len(missing_cats_landSlope) == 0 else "🔴"} missing categories {missing_cats_landSlope}')

landslope_new_cat = df_train[~df_train['LandSlope'].isin(landslope_cat_all)]
landslope_new_cat = landslope_new_cat['LandSlope'].unique().tolist()
print(f'{"🟢" if len(landslope_new_cat) == 0 else "🔴"} new categories or diferent name -> {landslope_new_cat}')
#Se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias tienen un orden definido, se procede a hacer ordinal-encoding
print('🟡 LandSlope: --> Ordinal-Encoding')
mapping_landslope = {'Gtl': 0, 'Mod': 1, 'Sev': 2}
display(mapping_landslope)
print('---'*20)

In [ ]:
# Neighborhood: Ubicación física dentro de la ciudad de Ames.
print("---" * 20)
cat_all = ["Blmngtn","Blueste","BrDale	","BrkSide","ClearCr","CollgCr","Crawfor","Edwards","Gilbert","IDOTRR	","MeadowV","Mitchel","Names	","NoRidge","NPkVill","NridgHt","NWAmes	","OldTown","SWISU	","Sawyer	","SawyerW","Somerst","StoneBr","Timber	","Veenker"]
neighborhood_cat_all = [cat_all.strip() for cat_all in cat_all]
neighborhood_cat_train = df_train["Neighborhood"].unique().tolist()
missing_cats_neighborhood = list(set(neighborhood_cat_all) - set(neighborhood_cat_train))
print(f'{"🟢" if len(missing_cats_neighborhood) == 0 else "🔴"} missing categories {missing_cats_neighborhood}')

neighborhood_new_cat = df_train[~df_train["Neighborhood"].isin(neighborhood_cat_all)]
neighborhood_new_cat = neighborhood_new_cat["Neighborhood"].unique().tolist()
print(
    f'{"🟢" if len(neighborhood_new_cat) == 0 else "🔴"} new categories or diferent name -> {neighborhood_new_cat}'
)
# No se tiene un orden definido en las categorias
# Pocas categorias
# Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#1. Se reemplaza NAmes por Names
df_train["Neighborhood"] = df_train["Neighborhood"].replace("NAmes", "Names")

neighborhood_cat_after_engineering = df_train["Neighborhood"].unique().tolist()
missing_cats_neighborhood = list(set(neighborhood_cat_all) - set(neighborhood_cat_after_engineering))
print(f'{"🟢" if len(missing_cats_neighborhood) == 0 else "🔴"} missing categories after engineering {missing_cats_neighborhood}')

print("🟡 Neighborhood: --> One-Hot-Encoding")
print("---" * 20)

In [ ]:
#Condition1: Proximidad a varias condiciones principales.
print('---'*20)
condition1_cat_all = [ "Artery", "Feedr", "Norm", "RRNn", "RRAn", "PosN", "PosA", "RRNe", "RRAe"]
condition1_cat_train = df_train['Condition1'].unique().tolist()
missing_cats_condition1 = list(set(condition1_cat_all) - set(condition1_cat_train))
print(f'{"🟢" if len(missing_cats_condition1) == 0 else "🔴"} missing categories {missing_cats_condition1}')

condition1_new_cat = df_train[~df_train['Condition1'].isin(condition1_cat_all)]
condition1_new_cat = condition1_new_cat['Condition1'].unique().tolist()
print(f'{"🟢" if len(condition1_new_cat) == 0 else "🔴"} new categories or diferent name -> {condition1_new_cat}')
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#Faltan categorias en el set de entrenamiento
print('🟡 Condition1: --> One-Hot-Encoding')
print('---'*20)

In [ ]:
#Condition2: Proximidad a varias condiciones secundarias.
print('---'*20)
condition2_cat_all = [ "Artery", "Feedr", "Norm", "RRNn", "RRAn", "PosN", "PosA", "RRNe", "RRAe"]
condition2_cat_train = df_train['Condition2'].unique().tolist()
missing_cats_condition2 = list(set(condition2_cat_all) - set(condition2_cat_train))
print(f'{"🟢" if len(missing_cats_condition2) == 0 else "🔴"} missing categories {missing_cats_condition2}')


condition2_new_cat = df_train[~df_train['Condition2'].isin(condition2_cat_all)]
condition2_new_cat = condition2_new_cat['Condition2'].unique().tolist()
print(f'{"🟢" if len(condition2_new_cat) == 0 else "🔴"} new categories or diferent name -> {condition2_new_cat}')
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#Faltan categorias en el set de entrenamiento
print('🟡 Condition2: --> One-Hot-Encoding')
print('---'*20)

In [ ]:
#BldgType : Tipo de vivienda.
print('---'*20)
bldgtype_cat_all = [ "1Fam", "2FmCon", "Duplx", "TwnhsE", "TwnhsI"]
bldgtype_cat_train = df_train['BldgType'].unique().tolist()
missing_cats_bldgType = list(set(bldgtype_cat_all) - set(bldgtype_cat_train))
print(f'{"🟢" if len(missing_cats_bldgType) == 0 else "🔴"} missing categories {missing_cats_bldgType}')

bldgtype_new_cat = df_train[~df_train['BldgType'].isin(bldgtype_cat_all)]
bldgtype_new_cat = bldgtype_new_cat['BldgType'].unique().tolist()
print(f'{"🟢" if len(bldgtype_new_cat) == 0 else "🔴"} new categories or diferent name -> {bldgtype_new_cat}')

#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#1. Convertir categorica Twnhs a TwnhsI
df_train['BldgType'] = df_train['BldgType'].replace('Twnhs', 'TwnhsI')
#2. Convertir categorica Duplx a Duplex
df_train['BldgType'] = df_train['BldgType'].replace('Duplex', 'Duplx')
#3. Convertir categorica 2fmCon a 2FmCon
df_train['BldgType'] = df_train['BldgType'].replace('2fmCon', '2FmCon')

bldgtype_cat_train = df_train['BldgType'].unique().tolist()
missing_cats_bldgType = list(set(bldgtype_cat_all) - set(bldgtype_cat_train))
print(f'{"🟢" if len(missing_cats_bldgType) == 0 else "🔴"} missing categories after engineering{missing_cats_bldgType}')

print('🟡 BldgType: --> One-Hot-Encoding')
print('---'*20)

In [ ]:
#HouseStyle : Estilo de la casa.
print('---'*20)
housestyle_cat_all = [ "1Story", "1.5Fin", "1.5Unf", "2Story", "2.5Fin", "2.5Unf", "SFoyer", "SLvl"]
housestyle_cat_train = df_train['HouseStyle'].unique().tolist()
missing_cats_houseStyle = list(set(housestyle_cat_all) - set(housestyle_cat_train))
print(f'{"🟢" if len(missing_cats_houseStyle) == 0 else "🔴"} missing categories {missing_cats_houseStyle}')

housestyle_new_cat = df_train[~df_train['HouseStyle'].isin(housestyle_cat_all)]
housestyle_new_cat = housestyle_new_cat['HouseStyle'].unique().tolist()
print(f'{"🟢" if len(housestyle_new_cat) == 0 else "🔴"} new categories or diferent name -> {housestyle_new_cat}')
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#@@Analizar la posibilidad de aplicar ordinal encoding
print('🟡 HouseStyle: --> One-Hot-Encoding')
print('---'*20)

In [ ]:
#RoofStyle : Tipo de techo.
print('---'*20)
roofstyle_cat_all = ['Flat', 'Gable', 'Gambrel', 'Hip', 'Mansard', 'Shed']
roofstyle_cat_train = df_train['RoofStyle'].unique().tolist()
missing_cats_roofStyle = list(set(roofstyle_cat_all) - set(roofstyle_cat_train))
print(f'{"🟢" if len(missing_cats_roofStyle) == 0 else "🔴"} missing categories {missing_cats_roofStyle}')

#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
roofstyle_new_cat = df_train[~df_train['RoofStyle'].isin(roofstyle_cat_all)]
roofstyle_new_cat = roofstyle_new_cat['RoofStyle'].unique().tolist()
print(f'{"🟢" if len(roofstyle_new_cat) == 0 else "🔴"} new categories or diferent name -> {roofstyle_new_cat}')
print('🟡 RoofStyle: --> One-Hot-Encoding')
print('---'*20)

In [ ]:
#RoofMatl : Material del techo.
print('---'*20)
roofmatl_cat_all = [ "ClyTile", "CompShg", "Membran", "Metal", "Roll", "Tar&Grv", "WdShake", "WdShngl"]
roofmatl_cat_train = df_train['RoofMatl'].unique().tolist()
missing_cats_roofMatl = list(set(roofmatl_cat_all) - set(roofmatl_cat_train))
print(f'{"🟢" if len(missing_cats_roofMatl) == 0 else "🔴"} missing categories {missing_cats_roofMatl}')

roofmatl_new_cat = df_train[~df_train['RoofMatl'].isin(roofmatl_cat_all)]
roofmatl_new_cat = roofmatl_new_cat['RoofMatl'].unique().tolist()
print(f'{"🟢" if len(roofmatl_new_cat) == 0 else "🔴"} new categories or diferent name -> {roofmatl_new_cat}')
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
print('🟡 RoofMatl: --> One-Hot-Encoding')
print('---'*20)

In [ ]:
#Exterior1st : Material exterior en la pared principal.
print('---'*20)
exterior1st_cat_all = [ "AsbShng", "AsphShn", "BrkComm", "BrkFace", "CBlock", "CemntBd", "HdBoard", "ImStucc", "MetalSd", "Other", "Plywood", "PreCast", "Stone", "Stucco", "VinylSd", "Wd Sdng", "WdShing"]
exterior1st_cat_train = df_train['Exterior1st'].unique().tolist()
missing_cats_exterior1st = list(set(exterior1st_cat_all) - set(exterior1st_cat_train))
print(f'{"🟢" if len(missing_cats_exterior1st) == 0 else "🔴"} missing categories {missing_cats_exterior1st}')

exterior1st_new_cat = df_train[~df_train['Exterior1st'].isin(exterior1st_cat_all)]
exterior1st_new_cat = exterior1st_new_cat['Exterior1st'].unique().tolist()
print(f'{"🟢" if len(exterior1st_new_cat) == 0 else "🔴"} new categories or diferent name -> {exterior1st_new_cat}')
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
print('🟡 Exterior1st: --> One-Hot-Encoding')
print('---'*20)

In [ ]:
#Exterior2nd : Material exterior en la pared secundaria.
print('---'*20)
exterior2nd_cat_all = [ "AsbShng", "AsphShn", "BrkComm", "BrkFace", "CBlock", "CemntBd", "HdBoard", "ImStucc", "MetalSd", "Other", "Plywood", "PreCast", "Stone", "Stucco", "VinylSd", "Wd Sdng", "WdShing"]
exterior2nd_cat_train = df_train['Exterior2nd'].unique().tolist()
missing_cats_exterior2nd = list(set(exterior2nd_cat_all) - set(exterior2nd_cat_train))
print(f'{"🟢" if len(missing_cats_exterior2nd) == 0 else "🔴"} missing categories {missing_cats_exterior2nd}')

exterior2nd_new_cat = df_train[~df_train['Exterior2nd'].isin(exterior2nd_cat_all)]
exterior2nd_new_cat = exterior2nd_new_cat['Exterior2nd'].unique().tolist()
print(f'{"🟢" if len(exterior2nd_new_cat) == 0 else "🔴"} new categories or diferent name -> {exterior2nd_new_cat}')
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#1. Se reemplaza Wd Shng por WdShing
df_train['Exterior2nd'] = df_train['Exterior2nd'].replace('Wd Shng', 'WdShing')
#2. Se reemplaza CmentBd por CmentBd por CemntBd
df_train['Exterior2nd'] = df_train['Exterior2nd'].replace('CmentBd', 'CemntBd')
#3. Se reemplaza Brk Cmn por BrkComm
df_train['Exterior2nd'] = df_train['Exterior2nd'].replace('Brk Cmn', 'BrkComm')

exterior2nd_cat_train = df_train['Exterior2nd'].unique().tolist()
missing_cats_exterior2nd = list(set(exterior2nd_cat_all) - set(exterior2nd_cat_train))
print(f'{"🟢" if len(missing_cats_exterior2nd) == 0 else "🔴"} missing categories after engineering {missing_cats_exterior2nd}')

print('🟡 Exterior2nd: --> One-Hot-Encoding')
print('---'*20)

In [ ]:
#ExterQual : Evaluación de la calidad del material exterior.
print('---'*20)
exterqual_cat_all = ['Ex', 'Gd', 'TA', 'Fa', 'Po']
exterqual_cat_train = df_train['ExterQual'].unique().tolist()
missing_cats_exterQual = list(set(exterqual_cat_all) - set(exterqual_cat_train))
print(f'{"🟢" if len(missing_cats_exterQual) == 0 else "🔴"} missing categories {missing_cats_exterQual}')

exterqual_new_cat = df_train[~df_train['ExterQual'].isin(exterqual_cat_all)]
exterqual_new_cat = exterqual_new_cat['ExterQual'].unique().tolist()
print(f'{"🟢" if len(exterqual_new_cat) == 0 else "🔴"} new categories or diferent name -> {exterqual_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_exterqual = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0}
print('🟡 ExterQual: --> Ordinal Encoding')
display(mapping_exterqual)
print('---'*20)

In [ ]:
#ExterCond : Evaluación de la condición del material exterior.
print('---'*20)
extercond_cat_all = ['Ex', 'Gd', 'TA', 'Fa', 'Po']
extercond_cat_train = df_train['ExterCond'].unique().tolist()
missing_cats_exterCond = list(set(extercond_cat_all) - set(extercond_cat_train))
print(f'{"🟢" if len(missing_cats_exterCond) == 0 else "🔴"} missing categories {missing_cats_exterCond}')

extercond_new_cat = df_train[~df_train['ExterCond'].isin(extercond_cat_all)]
extercond_new_cat = extercond_new_cat['ExterCond'].unique().tolist()
print(f'{"🟢" if len(extercond_new_cat) == 0 else "🔴"} new categories or diferent name -> {extercond_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_extercond = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0}
print('🟡 ExterCond: --> Ordinal Encoding')
print('---'*20)
display(mapping_extercond)

In [ ]:
#Foundation : Tipo de cimiento.
print('---'*20)
foundation_cat_all = [ "BrkTil", "CBlock", "PConc", "Slab", "Stone", "Wood"]
foundation_cat_train = df_train['Foundation'].unique().tolist()
missing_cats_fundation = list(set(foundation_cat_all) - set(foundation_cat_train))
print(f'{"🟢" if len(missing_cats_fundation) == 0 else "🔴"} missing categories {missing_cats_fundation}')

foundation_new_cat = df_train[~df_train['Foundation'].isin(foundation_cat_all)]
foundation_new_cat = foundation_new_cat['Foundation'].unique().tolist()
print(f'{"🟢" if len(foundation_new_cat) == 0 else "🔴"} new categories or diferent name -> {foundation_new_cat}')
#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
print('🟡 Foundation: --> One-Hot-Encoding')
print('---'*20)

In [ ]:
#BsmtQual : Evalúa la altura del sótano.
print('---'*20)
bsmtqual_cat_all = ['Ex', 'Gd', 'TA', 'Fa', 'Po', 'NA']
bsmtqual_cat_train = df_train['BsmtQual'].unique().tolist()
missing_cats_bsmtQual = list(set(bsmtqual_cat_all) - set(bsmtqual_cat_train))
print(f'{"🟢" if len(missing_cats_bsmtQual) == 0 else "🔴"} missing categories {missing_cats_bsmtQual}')

bsmtqual_new_cat = df_train[~df_train['BsmtQual'].isin(bsmtqual_cat_all)]
bsmtqual_new_cat = bsmtqual_new_cat['BsmtQual'].unique().tolist()
print(f'{"🟢" if len(bsmtqual_new_cat) == 0 else "🔴"} new categories or diferent name -> {bsmtqual_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_bsmtqual = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0, 'NA': 0}
print('🟡 BsmtQual: --> Ordinal Encoding')
print('---'*20)
display(mapping_bsmtqual)

In [ ]:
#BsmtCond : Evalúa la condición general del sótano
print('---'*20)
bsmtcond_cat_all = ['Ex', 'Gd', 'TA', 'Fa', 'Po', 'NA']
bsmtcond_cat_train = df_train['BsmtCond'].unique().tolist()
missing_cats_bsmtCond = list(set(bsmtcond_cat_all) - set(bsmtcond_cat_train))
print(f'{"🟢" if len(missing_cats_bsmtCond) == 0 else "🔴"} missing categories {missing_cats_bsmtCond}')

bsmtcond_new_cat = df_train[~df_train['BsmtCond'].isin(bsmtcond_cat_all)]
bsmtcond_new_cat = bsmtcond_new_cat['BsmtCond'].unique().tolist()
print(f'{"🟢" if len(bsmtcond_new_cat) == 0 else "🔴"} new categories or diferent name -> {bsmtcond_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_bsmtcond = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0, 'NA': 0}
print('🟡 BsmtCond: --> Ordinal Encoding')
print('---'*20)
display(mapping_bsmtcond)

In [ ]:
#BsmtExposure : Refleja la cantidad de exposición al sótano al aire exterior.
print('---'*20)
bsmtexposure_cat_all = ['Gd', 'Av', 'Mn', 'No', 'NA']
bsmtexposure_cat_train = df_train['BsmtExposure'].unique().tolist()
missing_cats_bsmtExposure = list(set(bsmtexposure_cat_all) - set(bsmtexposure_cat_train))
print(f'{"🟢" if len(missing_cats_bsmtExposure) == 0 else "🔴"} missing categories {missing_cats_bsmtExposure}')

bsmtexposure_new_cat = df_train[~df_train['BsmtExposure'].isin(bsmtexposure_cat_all)]
bsmtexposure_new_cat = bsmtexposure_new_cat['BsmtExposure'].unique().tolist()
print(f'{"🟢" if len(bsmtexposure_new_cat) == 0 else "🔴"} new categories or diferent name -> {bsmtexposure_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_bsmtexposure = {'Gd': 3, 'Av': 2, 'Mn': 1, 'No': 0, 'NA': 0}
print('🟡 BsmtExposure: --> Ordinal Encoding')
print('---'*20)
display(mapping_bsmtexposure)

In [ ]:
#BsmtFinType1 : Calidad del acabado del sótano.
print('---'*20)
bsmtfintype1_cat_all = ['GLQ', 'ALQ', 'BLQ', 'Rec', 'LwQ', 'Unf', 'NA']
bsmtfintype1_cat_train = df_train['BsmtFinType1'].unique().tolist()
missing_cats_bsmtFinType1 = list(set(bsmtfintype1_cat_all) - set(bsmtfintype1_cat_train))
print(f'{"🟢" if len(missing_cats_bsmtFinType1) == 0 else "🔴"} missing categories {missing_cats_bsmtFinType1}')

bsmtfintype1_new_cat = df_train[~df_train['BsmtFinType1'].isin(bsmtfintype1_cat_all)]
bsmtfintype1_new_cat = bsmtfintype1_new_cat['BsmtFinType1'].unique().tolist()
print(f'{"🟢" if len(bsmtfintype1_new_cat) == 0 else "🔴"} new categories or diferent name -> {bsmtfintype1_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_bsmtfintype1 = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'NA': 0}
print('🟡 BsmtFinType1: --> Ordinal Encoding')
print('---'*20)
display(mapping_bsmtfintype1)

In [ ]:
#BsmtFinType2 : Calidad del acabado del sótano (si hay dos áreas terminadas).
print('---'*20)
bsmtfintype2_cat_all = ['GLQ', 'ALQ', 'BLQ', 'Rec', 'LwQ', 'Unf', 'NA']
bsmtfintype2_cat_train = df_train['BsmtFinType2'].unique().tolist()
missing_cats_bsmtFinType2 = list(set(bsmtfintype2_cat_all) - set(bsmtfintype2_cat_train))
print(f'{"🟢" if len(missing_cats_bsmtFinType2) == 0 else "🔴"} missing categories {missing_cats_bsmtFinType2}')

bsmtfintype2_new_cat = df_train[~df_train['BsmtFinType2'].isin(bsmtfintype2_cat_all)]
bsmtfintype2_new_cat = bsmtfintype2_new_cat['BsmtFinType2'].unique().tolist()
print(f'{"🟢" if len(bsmtfintype2_new_cat) == 0 else "🔴"} new categories or diferent name -> {bsmtfintype2_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_bsmtfintype2 = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'NA': 0}
print('🟡 BsmtFinType2: --> Ordinal Encoding')
print('---'*20)
display(mapping_bsmtfintype2)

In [ ]:
#Heating : Tipo de calefacción.
print('---'*20)
heating_cat_all = ['Floor', 'GasA', 'GasW', 'Grav', 'OthW', 'Wall']
heating_cat_train = df_train['Heating'].unique().tolist()
missing_cats_heating = list(set(heating_cat_all) - set(heating_cat_train))
print(f'{"🟢" if len(missing_cats_heating) == 0 else "🔴"} missing categories {missing_cats_heating}')

heating_new_cat = df_train[~df_train['Heating'].isin(heating_cat_all)]
heating_new_cat = heating_new_cat['Heating'].unique().tolist()
print(f'{"🟢" if len(heating_new_cat) == 0 else "🔴"} new categories or diferent name -> {heating_new_cat}')

#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
print('🟡 Heating: --> One-Hot-Encoding')
print('---'*20)

In [ ]:
#HeatingQC : Evaluación de la calidad y el estado del sistema de calefacción.
print('---'*20)
heatingqc_cat_all = ['Ex', 'Gd', 'TA', 'Fa', 'Po']
heatingqc_cat_train = df_train['HeatingQC'].unique().tolist()
missing_cats_heatingQC = list(set(heatingqc_cat_all) - set(heatingqc_cat_train))
print(f'{"🟢" if len(missing_cats_heatingQC) == 0 else "🔴"} missing categories {missing_cats_heatingQC}')

heatingqc_new_cat = df_train[~df_train['HeatingQC'].isin(heatingqc_cat_all)]
heatingqc_new_cat = heatingqc_new_cat['HeatingQC'].unique().tolist()
print(f'{"🟢" if len(heatingqc_new_cat) == 0 else "🔴"} new categories or diferent name -> {heatingqc_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_heatingqc = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0}
print('🟡 HeatingQC: --> Ordinal Encoding')
print('---'*20)
display(mapping_heatingqc)

In [ ]:
#CentralAir : Aire acondicionado central.
print('---'*20)
centralair_cat_all = ['Y', 'N']
centralair_cat_train = df_train['CentralAir'].unique().tolist()
missing_cats_centralAir = list(set(centralair_cat_all) - set(centralair_cat_train))
print(f'{"🟢" if len(missing_cats_centralAir) == 0 else "🔴"} missing categories {missing_cats_centralAir}')

centralair_new_cat = df_train[~df_train['CentralAir'].isin(centralair_cat_all)]
centralair_new_cat = centralair_new_cat['CentralAir'].unique().tolist()
print(f'{"🟢" if len(centralair_new_cat) == 0 else "🔴"} new categories or diferent name -> {centralair_new_cat}')

#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
print('🟡 CentralAir: --> One-Hot-Encoding')
print('---'*20)

In [ ]:
#Electrical : Tipo de sistema eléctrico.
print('---'*20)
electrical_cat_all = ['SBrkr', 'FuseA', 'FuseF', 'FuseP', 'Mix']
electrical_cat_train = df_train['Electrical'].unique().tolist()
missing_cats_electrical = list(set(electrical_cat_all) - set(electrical_cat_train))
print(f'{"🟢" if len(missing_cats_electrical) == 0 else "🔴"} missing categories {missing_cats_electrical}')

electrical_new_cat = df_train[~df_train['Electrical'].isin(electrical_cat_all)]
electrical_new_cat = electrical_new_cat['Electrical'].unique().tolist()
print(f'{"🟢" if len(electrical_new_cat) == 0 else "🔴"} new categories or diferent name -> {electrical_new_cat}')

#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
#@@Puede ser aplicado con ordinal encoding

print('🟡 Electrical: --> One-Hot-Encoding')
print('---'*20)

In [ ]:
#KitchenQual : Calidad de la cocina.
print('---'*20)
kitchenqual_cat_all = ['Ex', 'Gd', 'TA', 'Fa', 'Po']
kitchenqual_cat_train = df_train['KitchenQual'].unique().tolist()
missing_cats_kitchenQual = list(set(kitchenqual_cat_all) - set(kitchenqual_cat_train))
print(f'{"🟢" if len(missing_cats_kitchenQual) == 0 else "🔴"} missing categories {missing_cats_kitchenQual}')

kitchenqual_new_cat = df_train[~df_train['KitchenQual'].isin(kitchenqual_cat_all)]
kitchenqual_new_cat = kitchenqual_new_cat['KitchenQual'].unique().tolist()
print(f'{"🟢" if len(kitchenqual_new_cat) == 0 else "🔴"} new categories or diferent name -> {kitchenqual_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_kitchenqual = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0}
print('🟡 KitchenQual: --> Ordinal Encoding')
print('---'*20)
display(mapping_kitchenqual)

In [ ]:
#Functional : Evaluación de la funcionalidad.
print('---'*20)
functional_cat_all = ['Typ', 'Min1', 'Min2', 'Mod', 'Maj1', 'Maj2', 'Sev', 'Sal']
functional_cat_train = df_train['Functional'].unique().tolist()
missing_cats_functional = list(set(functional_cat_all) - set(functional_cat_train))
print(f'{"🟢" if len(missing_cats_functional) == 0 else "🔴"} missing categories {missing_cats_functional}')

functional_new_cat = df_train[~df_train['Functional'].isin(functional_cat_all)]
functional_new_cat = functional_new_cat['Functional'].unique().tolist()
print(f'{"🟢" if len(functional_new_cat) == 0 else "🔴"} new categories or diferent name -> {functional_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
#Podria ser probado con one-hot-encoding
mapping_functional = {'Typ': 7, 'Min1': 6, 'Min2': 5, 'Mod': 4, 'Maj1': 3, 'Maj2': 2, 'Sev': 1, 'Sal': 0}
print('🟡 Functional: --> Ordinal Encoding')
print('---'*20)
display(mapping_functional)

In [ ]:
#GarageType : Ubicación del garaje.
print('---'*20)
garagetype_cat_all = [ "2Types", "Attchd", "Basment", "BuiltIn", "CarPort", "Detchd", "NA"]
garagetype_cat_train = df_train['GarageType'].unique().tolist()
missing_cats_garageType = list(set(garagetype_cat_all) - set(garagetype_cat_train))
print(f'{"🟢" if len(missing_cats_garageType) == 0 else "🔴"} missing categories {missing_cats_garageType}')

garagetype_new_cat = df_train[~df_train['GarageType'].isin(garagetype_cat_all)]
garagetype_new_cat = garagetype_new_cat['GarageType'].unique().tolist()
print(f'{"🟢" if len(garagetype_new_cat) == 0 else "🔴"} new categories or diferent name -> {garagetype_new_cat}')

#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding

print('🟡 GarageType: --> One-Hot-Encoding')
print('---'*20)

In [ ]:
#GarageFinish : Acabado interior del garaje.
print('---'*20)
garagefinish_cat_all = ['Fin', 'RFn', 'Unf', 'NA']
garagefinish_cat_train = df_train['GarageFinish'].unique().tolist()
missing_cats_garageFinish = list(set(garagefinish_cat_all) - set(garagefinish_cat_train))
print(f'{"🟢" if len(missing_cats_garageFinish) == 0 else "🔴"} missing categories {missing_cats_garageFinish}')

garagefinish_new_cat = df_train[~df_train['GarageFinish'].isin(garagefinish_cat_all)]
garagefinish_new_cat = garagefinish_new_cat['GarageFinish'].unique().tolist()
print(f'{"🟢" if len(garagefinish_new_cat) == 0 else "🔴"} new categories or diferent name -> {garagefinish_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_garagefinish = {'Fin': 3, 'RFn': 2, 'Unf': 1, 'NA': 0}
print('🟡 GarageFinish: --> Ordinal Encoding')
print('---'*20)
display(mapping_garagefinish)

In [ ]:
#GarageQual : Evaluación de la calidad del garaje.
print('---'*20)
garagequal_cat_all = ['Ex', 'Gd', 'TA', 'Fa', 'Po', 'NA']
garagequal_cat_train = df_train['GarageQual'].unique().tolist()
missing_cats_garageQual = list(set(garagequal_cat_all) - set(garagequal_cat_train))
print(f'{"🟢" if len(missing_cats_garageQual) == 0 else "🔴"} missing categories {missing_cats_garageQual}')

garagequal_new_cat = df_train[~df_train['GarageQual'].isin(garagequal_cat_all)]
garagequal_new_cat = garagequal_new_cat['GarageQual'].unique().tolist()
print(f'{"🟢" if len(garagequal_new_cat) == 0 else "🔴"} new categories or diferent name -> {garagequal_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_garagequal = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0, 'NA': 0}
print('🟡 GarageQual: --> Ordinal Encoding')
print('---'*20)
display(mapping_garagequal)

In [ ]:
#GarageCond : Evaluación de la condición del garaje.
print('---'*20)
garagecond_cat_all = ['Ex', 'Gd', 'TA', 'Fa', 'Po', 'NA']
garagecond_cat_train = df_train['GarageCond'].unique().tolist()
missing_cats_garageCond = list(set(garagecond_cat_all) - set(garagecond_cat_train))
print(f'{"🟢" if len(missing_cats_garageCond) == 0 else "🔴"} missing categories {missing_cats_garageCond}')

garagecond_new_cat = df_train[~df_train['GarageCond'].isin(garagecond_cat_all)]
garagecond_new_cat = garagecond_new_cat['GarageCond'].unique().tolist()
print(f'{"🟢" if len(garagecond_new_cat) == 0 else "🔴"} new categories or diferent name -> {garagecond_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_garagecond = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0, 'NA': 0}
print('🟡 GarageCond: --> Ordinal Encoding')
print('---'*20)
display(mapping_garagecond)

In [ ]:
#PavedDrive : Tipo de camino pavimentado.
print('---'*20)
paveddrive_cat_all = ['Y', 'P', 'N']
paveddrive_cat_train = df_train['PavedDrive'].unique().tolist()
missing_cats_pavedDrive = list(set(paveddrive_cat_all) - set(paveddrive_cat_train))
print(f'{"🟢" if len(missing_cats_pavedDrive) == 0 else "🔴"} missing categories {missing_cats_pavedDrive}')

paveddrive_new_cat = df_train[~df_train['PavedDrive'].isin(paveddrive_cat_all)]
paveddrive_new_cat = paveddrive_new_cat['PavedDrive'].unique().tolist()
print(f'{"🟢" if len(paveddrive_new_cat) == 0 else "🔴"} new categories or diferent name -> {paveddrive_new_cat}')

#Se define como ordinal encoding
#Pocas categorias
mapping_paveddrive = {'Y': 1, 'P': 0, 'N': 0}
print('🟡 PavedDrive: --> Ordinal Encoding')
print('---'*20)
display(mapping_paveddrive)

In [ ]:
#SaleType : Tipo de venta.
print('---'*20)
saletype_cat_all = ['WD', 'CWD', 'VWD', 'New', 'COD', 'Con', 'ConLw', 'ConLI', 'ConLD', 'Oth']
saletype_cat_train = df_train['SaleType'].unique().tolist()
missing_cats_saleType = list(set(saletype_cat_all) - set(saletype_cat_train))
print(f'{"🟢" if len(missing_cats_saleType) == 0 else "🔴"} missing categories {missing_cats_saleType}')

saletype_new_cat = df_train[~df_train['SaleType'].isin(saletype_cat_all)]
saletype_new_cat = saletype_new_cat['SaleType'].unique().tolist()
print(f'{"🟢" if len(saletype_new_cat) == 0 else "🔴"} new categories or diferent name -> {saletype_new_cat}')

#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
print('🟡 SaleType: --> One-Hot-Encoding')
print('---'*20)

In [ ]:
#SaleCondition : Condición de la venta.
print('---'*20)
salecondition_cat_all = ['Normal', 'Abnorml', 'AdjLand', 'Alloca', 'Family', 'Partial']
salecondition_cat_train = df_train['SaleCondition'].unique().tolist()
missing_cats_saleCondition = list(set(salecondition_cat_all) - set(salecondition_cat_train))
print(f'{"🟢" if len(missing_cats_saleCondition) == 0 else "🔴"} missing categories {missing_cats_saleCondition}')

salecondition_new_cat = df_train[~df_train['SaleCondition'].isin(salecondition_cat_all)]
salecondition_new_cat = salecondition_new_cat['SaleCondition'].unique().tolist()
print(f'{"🟢" if len(salecondition_new_cat) == 0 else "🔴"} new categories or diferent name -> {salecondition_new_cat}')

#No se tiene un orden definido en las categorias
#Pocas categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
print('🟡 SaleCondition: --> One-Hot-Encoding')
print('---'*20)

---
### 🔢 Análisis de Features Numéricas

In [ ]:
#MSSubClass: Clase de construcción.
#Tiene valores numericos pero son categoricos, por lo que se trata como variable categorica
#Los valores numericos representan diferentes tipos de viviendas.
print('---'*20)
msubclass_cat_all = [20, 30, 40, 45, 50, 60, 70, 75, 80, 85, 90, 120, 150, 160, 180, 190]
msubclass_cat_train = df_train['MSSubClass'].unique().tolist()
missing_cats_msSubClass = list(set(msubclass_cat_all) - set(msubclass_cat_train))
print(f'{"🟢" if len(missing_cats_msSubClass) == 0 else "🔴"} missing categories {missing_cats_msSubClass}')
msubclass_new_cat = df_train[~df_train['MSSubClass'].isin(msubclass_cat_all)]
msubclass_new_cat = msubclass_new_cat['MSSubClass'].unique().tolist()
print(f'{"🟢" if len(msubclass_new_cat) == 0 else "🔴"} new categories or diferent name -> {msubclass_new_cat}')
#No se tiene un orden definido en las categorias
#Visto que las categorias no tienen un orden definido, se procede a hacer one-hot-encoding
print('🟡 MSSubClass: --> One-Hot-Encoding')

In [ ]:
#LotArea: Tamaño del lote en pies cuadrados.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
resumen_analyst(df_train, 'LotArea', False)

#Los datos presentan una distribución sesgada a la derecha, con una cola larga hacia valores más altos de LotArea.
#Proximo a analizar los valores atipicos

In [ ]:
#OverallQual: Califica la calidad general del material y los acabados de la casa.
print('---'*20)
overallqual_cat_all = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
overallqual_cat_train = df_train['OverallQual'].unique().tolist()
missing_cats = list(set(overallqual_cat_all) - set(overallqual_cat_train))
print(f'{"🟢" if len(missing_cats) == 0 else "🔴"} missing categories {missing_cats}')
overallqual_new_cat = df_train[~df_train['OverallQual'].isin(overallqual_cat_all)]
overallqual_new_cat = overallqual_new_cat['OverallQual'].unique().tolist()
print(f'{"🟢" if len(overallqual_new_cat) == 0 else "🔴"} new categories or diferent name -> {overallqual_new_cat}')
#Se define como ordinal encoding
#Pocas categorias
mapping_overallqual = {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10}
print('🟡 OverallQual: --> Ordinal Encoding')
print('---'*20)
display(mapping_overallqual)


In [ ]:
#OverallCond: Califica la condición general de la casa
print('---'*20)
overallcond_cat_all = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
overallcond_cat_train = df_train['OverallCond'].unique().tolist()
missing_cats_overallCond = list(set(overallcond_cat_all) - set(overallcond_cat_train))
print(f'{"🟢" if len(missing_cats_overallCond) == 0 else "🔴"} missing categories {missing_cats_overallCond}')
overallcond_new_cat = df_train[~df_train['OverallCond'].isin(overallcond_cat_all)]
overallcond_new_cat = overallcond_new_cat['OverallCond'].unique().tolist()
print(f'{"🟢" if len(overallcond_new_cat) == 0 else "🔴"} new categories or diferent name -> {overallcond_new_cat}')
#Se define como ordinal encoding
#Pocas categorias
mapping_overallcond = {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10}
print('🟡 OverallCond: --> Ordinal Encoding')
print('---'*20)
display(mapping_overallcond)

In [ ]:

#YearBuilt: Fecha de construcción original.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 YearBuilt: --> No Encoding applied')
resumen_analyst(df_train, 'YearBuilt', False)
#Los datos presentan una distribución aproximadamente normal, con una ligera concentración alrededor de los años más recientes.

In [ ]:
#YearRemodAdd: Fecha de remodelación (igual que la fecha de construcción si no hubo remodelaciones o adiciones).
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 YearRemodAdd: --> No Encoding applied')
resumen_analyst(df_train, 'YearRemodAdd', False)
#Los datos presentan una distribución aproximadamente normal, con una ligera concentración alrededor de los años más recientes.
#Próximo a analizar los valores atipicos

In [ ]:
#MasVnrArea: Área de revestimiento de mampostería en pies cuadrados.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 MasVnrArea: --> No Encoding applied')
resumen_analyst(df_train, 'MasVnrArea', False)
#Los datos presentan una distribución sesgada a la derecha, con una cola larga hacia valores más altos de MasVnrArea.
#Próximo a analizar los valores atipicos

In [ ]:
#BsmtFinSF1: Pies cuadrados terminados de tipo 1.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 BsmtFinSF1: --> No Encoding applied')
resumen_analyst(df_train, 'BsmtFinSF1', False)
#Los datos presentan una distribución sesgada a la derecha, con una cola larga hacia valores más altos de BsmtFinSF1.
#Proximo a analizar los valores atipicos

In [ ]:
#BsmtFinSF2: Pies cuadrados terminados de tipo 2.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 BsmtFinSF2: --> No Encoding applied')
resumen_analyst(df_train, 'BsmtFinSF2', False)
#Los datos presentan una distribución sesgada a la derecha, con una cola larga hacia valores más altos de BsmtFinSF2.
#Próximo a analizar los valores atípicos

In [ ]:
#BsmtUnfSF: Pies cuadrados sin terminar del área del sótano.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 BsmtUnfSF: --> No Encoding applied')
resumen_analyst(df_train, 'BsmtUnfSF', False)
#Los datos presentan una distribución sesgada a la derecha, con una cola larga hacia valores más altos de BsmtUnfSF.
#Próximo a analizar los valores atípicos

In [ ]:
#TotalBsmtSF: Pies cuadrados totales del área del sótano.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 TotalBsmtSF: --> No Encoding applied')
resumen_analyst(df_train, 'TotalBsmtSF', False)
#Los datos presentan una distribución sesgada a la derecha, con una cola larga hacia valores más altos de TotalBsmtSF.
#Próximo a analizar los valores atípicos

In [ ]:
#1stFlrSF: Pies cuadrados del primer piso.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 1stFlrSF: --> No Encoding applied')
resumen_analyst(df_train, '1stFlrSF', False)
#Los datos presentan una distribución sesgada a la derecha, con una cola larga hacia valores más altos de 1stFlrSF.
#Próximo a analizar los valores atípicos

In [ ]:
#2ndFlrSF: Pies cuadrados del segundo piso.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 2ndFlrSF: --> No Encoding applied')
resumen_analyst(df_train, '2ndFlrSF', False)
#Los datos presentan una distribución sesgada a la derecha, con una cola larga hacia valores más altos de 2ndFlrSF.
#Próximo a analizar los valores atípicos

In [ ]:
#LowQualFinSF: Pies cuadrados terminados de baja calidad (todos los pisos).
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 LowQualFinSF: --> No Encoding applied')
resumen_analyst(df_train, 'LowQualFinSF', False)
#Los datos presentan una distribución sesgada a la derecha, con una cola larga hacia valores más altos de LowQualFinSF.
#Próximo a analizar los valores atípicos

In [ ]:
#GrLivArea: Pies cuadrados de área habitable sobre el nivel del suelo.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 GrLivArea: --> No Encoding applied')
resumen_analyst(df_train, 'GrLivArea', False)
#Los datos presentan una distribución sesgada a la derecha, con una cola larga hacia valores más altos de GrLivArea.
#Próximo a analizar los valores atípicos

In [ ]:
#BsmtFullBath: Baños completos en el sótano.
print('---'*20)
#Es una variable numérica discreta, no se aplica codificación
print('🟡 BsmtFullBath: --> No Encoding applied')
resumen_analyst(df_train, 'BsmtFullBath', False)
#La mayoría de las casas tienen 0 o 1 baño completo en el sótano, con una disminución significativa en la frecuencia a medida que aumenta el número de baños.
#Próximo a analizar los valores atípicos

In [ ]:
#BsmtHalfBath: Medios baños en el sótano.
print('---'*20)
#Es una variable numérica discreta, no se aplica codificación
print('🟡 BsmtHalfBath: --> No Encoding applied')
resumen_analyst(df_train, 'BsmtHalfBath', False)
#La mayoría de las casas tienen 0 medios baños en el sótano, con una disminución significativa en la frecuencia a medida que aumenta el número de medios baños.
#Próximo a analizar los valores atípicos

In [ ]:
#FullBath: Baños completos sobre el nivel del suelo.
print('---'*20)
#Es una variable numérica discreta, no se aplica codificación
print('🟡 FullBath: --> No Encoding applied')
resumen_analyst(df_train, 'FullBath', False)
#La mayoría de las casas tienen 1 o 2 baños completos sobre el nivel del suelo, con una disminución significativa en la frecuencia a medida que aumenta el número de baños.
#Próximo a analizar los valores atípicos

In [ ]:
#HalfBath: Medios baños sobre el nivel del suelo.
print('---'*20)
#Es una variable numérica discreta, no se aplica codificación
print('🟡 HalfBath: --> No Encoding applied')
resumen_analyst(df_train, 'HalfBath', False)
#La mayoría de las casas tienen 0 o 1 medio baño sobre el nivel del suelo, con una disminución significativa en la frecuencia a medida que aumenta el número de medios baños.
#Próximo a analizar los valores atípicos
#Clai

In [ ]:
#Bedroom || BedroomAbvGr: Dormitorios sobre el nivel del suelo (NO incluye dormitorios en el sótano).
print('---'*20)
#Es una variable numérica discreta, no se aplica codificación
print('🟡 BedroomAbvGr: --> No Encoding applied')
resumen_analyst(df_train, 'BedroomAbvGr', False)
#La mayoría de las casas tienen entre 2 y 4 dormitorios sobre el nivel del suelo
#Próximo a analizar los valores atípicos

In [ ]:
#Kitchen || KitchenAbvGr: Cocinas sobre el nivel del suelo.
print('---'*20)
#Es una variable numérica discreta, no se aplica codificación
print('🟡 KitchenAbvGr: --> No Encoding applied')
resumen_analyst(df_train, 'KitchenAbvGr', False)
#La mayoría de las casas tienen 1 cocina sobre el nivel del suelo, con una disminución significativa en la frecuencia a medida que aumenta el número de cocinas.
#Próximo a analizar los valores atípicos

In [ ]:
#TotRmsAbvGrd: Total de habitaciones sobre el nivel del suelo (no incluye baños).
print('---'*20)
#Es una variable numérica discreta, no se aplica codificación
print('🟡 TotRmsAbvGrd: --> No Encoding applied')
resumen_analyst(df_train, 'TotRmsAbvGrd', False)
#La mayoría de las casas tienen entre 5 y 8 habitaciones sobre el nivel del suelo.
#Próximo a analizar los valores atípicos

In [ ]:
#Fireplaces: Número de chimeneas.
print('---'*20)
#Es una variable numérica discreta, no se aplica codificación
print('🟡 Fireplaces: --> No Encoding applied')
resumen_analyst(df_train, 'Fireplaces', False)
#La mayoría de las casas tienen entre 0 y 2 chimeneas, con una disminución significativa en la frecuencia a medida que aumenta el número de chimeneas.
#Próximo a analizar los valores atípicos

In [ ]:
#GarageYrBlt: Año de construcción del garaje.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 GarageYrBlt: --> No Encoding applied')
resumen_analyst(df_train, 'GarageYrBlt', False)
#Los datos presentan una distribución aproximadamente normal, con una ligera concentración alrededor de los años más recientes.
#Próximo a analizar los valores atípicos

In [ ]:
#GarageCars: Tamaño del garaje en capacidad de coches.
print('---'*20)
#Es una variable numérica discreta, no se aplica codificación
print('🟡 GarageCars: --> No Encoding applied')
resumen_analyst(df_train, 'GarageCars', False)
#La mayoría de las casas tienen entre 1 y 3 plazas de garaje, con una disminución significativa en la frecuencia a medida que aumenta la capacidad del garaje.
#Próximo a analizar los valores atípicos

In [ ]:
#GarageArea: Tamaño del garaje en pies cuadrados.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 GarageArea: --> No Encoding applied')
resumen_analyst(df_train, 'GarageArea', False)
#Los datos presentan una distribución sesgada a la derecha, con una cola larga hacia valores más altos de GarageArea.
#Próximo a analizar los valores atípicos

In [ ]:
#WoodDeckSF: Área de la terraza de madera en pies cuadrados.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 WoodDeckSF: --> No Encoding applied')
resumen_analyst(df_train, 'WoodDeckSF', False)
#Los datos presentan una distribución sesgada a la derecha, con una cola larga hacia valores más altos de WoodDeckSF.
#Próximo a analizar los valores atípicos

In [ ]:
#OpenPorchSF: Área del porche abierto en pies cuadrados.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 OpenPorchSF: --> No Encoding applied')
resumen_analyst(df_train, 'OpenPorchSF', False)
#Los datos presentan una distribución sesgada a la derecha, con una cola larga hacia valores más altos de OpenPorchSF.
#Próximo a analizar los valores atípicos

In [ ]:
#EnclosedPorch: Área del porche cerrado en pies cuadrados.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 EnclosedPorch: --> No Encoding applied')
resumen_analyst(df_train, 'EnclosedPorch', False)
#Los datos presentan una distribución sesgada a la derecha, con una cola larga hacia valores más altos de EnclosedPorch.
#Próximo a analizar los valores atípicos

In [ ]:
#3SsnPorch: Área del porche de tres estaciones en pies cuadrados.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 3SsnPorch: --> No Encoding applied')
resumen_analyst(df_train, '3SsnPorch', False)
#Los datos presentan una distribución sesgada a la derecha, con una cola larga hacia valores más altos de 3SsnPorch.
#Próximo a analizar los valores atípicos

In [ ]:
#ScreenPorch: Área del porche con mosquitero en pies cuadrados.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 ScreenPorch: --> No Encoding applied')
resumen_analyst(df_train, 'ScreenPorch', False)
#Los datos presentan una distribución sesgada a la derecha, con una cola larga hacia valores más altos de ScreenPorch.
#Próximo a analizar los valores atípicos

In [ ]:
#PoolArea: Área de la piscina en pies cuadrados.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 PoolArea: --> No Encoding applied')
resumen_analyst(df_train, 'PoolArea', False)
#La mayoría de las casas tienen un área de piscina de 0 pies cuadrados, lo que indica que no tienen piscina.
#Próximo a analizar los valores atípicos

In [ ]:
#MiscVal: Valor en dólares de la característica miscelánea.
print('---'*20)
#Es una variable numérica continua, no se aplica codificación
print('🟡 MiscVal: --> No Encoding applied')
resumen_analyst(df_train, 'MiscVal', False)
#La mayoría de las casas tienen un valor misceláneo de 0 dólares, lo que indica que no tienen características misceláneas de valor significativo.
#Próximo a analizar los valores atípicos

In [ ]:
#MoSold: Mes de venta (MM).
print('---'*20)
#Es una variable numérica discreta, no se aplica codificación
print('🟡 MoSold: --> No Encoding applied')
resumen_analyst(df_train, 'MoSold', False)
#Las ventas parecen estar distribuidas a lo largo de todo el año, con picos en los meses de mayo, junio y julio.
#Próximo a analizar los valores atípicos

In [ ]:
#YrSold: Año de venta (AAAA).
print('---'*20)
#Es una variable numérica discreta, no se aplica codificación
print('🟡 YrSold: --> No Encoding applied')
resumen_analyst(df_train, 'YrSold', False)
#Las ventas parecen estar distribuidas de manera relativamente uniforme a lo largo de los años 2006 a 2010.
#Próximo a analizar los valores atípicos